In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# 讀取最終版資料
df = pd.read_csv('../data/processed/yrbs_final.csv')
print(f"✅ 資料載入成功，樣本數：{len(df)}")

✅ 資料載入成功，樣本數：12669


「2007 年美國高中生中有飲酒習慣（Success）的比例，是否顯著不同於基準值 0.35？

In [2]:
p0 = 0.35  
successes = df['alcohol_binary'].sum()
n_obs_alc = len(df['alcohol_binary'].dropna())

# 執行單一樣本 Z 檢定
stat_z, p_val_z = proportions_ztest(successes, n_obs_alc, value=p0)
# 計算 95% 信賴區間
ci_low_p, ci_high_p = proportion_confint(successes, n_obs_alc, alpha=0.05)

print(f"樣本比例: {successes/n_obs_alc:.4f}")
print(f"P-value: {p_val_z:.4e}")
print(f"95% CI: [{ci_low_p:.4f}, {ci_high_p:.4f}]")

樣本比例: 0.4517
P-value: 3.8041e-117
95% CI: [0.4431, 0.4604]


In [3]:
mu0 = 68.0  # 老師指定的基準值
weight_data = df['HowMuchDoYouWeighWithoutShoesInKG'].dropna()

# 執行單一樣本 T 檢定
t_stat, p_val_t = stats.ttest_1samp(weight_data, popmean=mu0)
# 計算 95% 信賴區間
ci_mean = stats.t.interval(0.95, len(weight_data)-1, loc=weight_data.mean(), scale=stats.sem(weight_data))

print(f"樣本平均數: {weight_data.mean():.2f} kg")
print(f"P-value: {p_val_t:.4e}")
print(f"95% CI: [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]")

樣本平均數: 68.43 kg
P-value: 6.0993e-03
95% CI: [68.12, 68.73]


數值摘要：

酒精使用比例為 45.17% ，信賴區間為 [0.4431, 0.4604]。

平均體重為 68.55 kg ，信賴區間為 [68.26, 68.84]。

信賴區間含意：

比例分析：我們有 95% 的信心，全體美國高中生的飲酒比例落在 0.4431 到 0.4604 之間，此區間完全不包含基準值 0.35。

平均數分析：體重信賴區間不包含 68.0 kg，代表母體平均值有極大機率高於此基準。

假設檢定結論：

由於兩者的 P 值均遠小於 0.05，我們拒絕虛無假設。這代表 2007 年青少年的飲酒比例與平均體重，均顯著不同於老師提供的官方基準值。

一致性觀察：推論結果與 EDA 觀察高度一致。在 EDA 中我們看到飲酒比例接近 45%，且體重分佈受高端離群值影響而略高於基準值 68 kg。


詮釋：體重分析中偵測到 447 筆離群值 ，雖然保留資料是為了生理多樣性，但這些極端高值確實拉高了平均數，若目標群體不含極端過重者，結果可能不同。

In [4]:
import pandas as pd
import os

# --- 1. 整理推論統計數據 ---
inference_results = [
    {
        "Analysis_Type": "Proportion (Alcohol Use)",
        "Variable": "alcohol_binary",
        "n": n_obs_alc,
        "Sample_Stat": round(successes/n_obs_alc, 4),
        "Benchmark": 0.35, # 老師規定的 p0
        "Test_Statistic": round(stat_z, 4),
        "P_value": f"{p_val_z:.4e}",
        "CI_Lower": round(ci_low_p, 4),
        "CI_Upper": round(ci_high_p, 4)
    },
    {
        "Analysis_Type": "Mean (Weight)",
        "Variable": "HowMuchDoYouWeighWithoutShoesInKG",
        "n": len(weight_data),
        "Sample_Stat": round(weight_data.mean(), 2),
        "Benchmark": 68.0, # 老師規定的 mu0
        "Test_Statistic": round(t_stat, 4),
        "P_value": f"{p_val_t:.4e}",
        "CI_Lower": round(ci_mean[0], 2),
        "CI_Upper": round(ci_mean[1], 2)
    }
]

# --- 2. 轉換為 DataFrame ---
df_inference_summary = pd.DataFrame(inference_results)

# --- 3. 儲存表格 ---
os.makedirs('../outputs/tables/', exist_ok=True)
df_inference_summary.to_csv('../outputs/tables/inference_summary_table.csv', index=False, encoding='utf-8-sig')

print("✅ 推論統計摘要表已成功存入: ../outputs/tables/inference_summary_table.csv")
display(df_inference_summary)

✅ 推論統計摘要表已成功存入: ../outputs/tables/inference_summary_table.csv


,Analysis_Type,Variable,n,Sample_Stat,Benchmark,Test_Statistic,P_value,CI_Lower,CI_Upper
0,Proportion (Alcohol Use),alcohol_binary,12669,0.4517,0.35,23.0088,3.8041e-117,0.4431,0.4604
1,Mean (Weight),HowMuchDoYouWeighWithoutShoesInKG,11843,68.4300,68.00,2.7429,6.0993e-03,68.1200,68.7300
